# DentaMind AI - 5-seed BASE ensemble, validation only (Kaggle GPU)

Protocol: `reports/ensemble_protocol.md`. Mean of the five BASE members' softmax probabilities; no tuning. The DENTEX **test split is
never loaded**. This runs the same script as locally (`kaggle/ensemble_eval.py`, embedded below and hash-checked) in the Kaggle
environment that produced the BASE numbers, because the local re-evaluation failed the reproduction gate (software differences).

**Before running:** GPU T4, Internet On, Add Input -> (1) the private bundle dataset, (2) Notebook output "DentaMind external data
training".

In [ ]:
# ---- 1. inputs: bundle code + previous run's checkpoints; write the ensemble script ----------------------------------
import glob, hashlib, json, sys, time
from pathlib import Path

def one(pattern):
    hits = sorted(glob.glob("/kaggle/input/**/" + pattern, recursive=True))
    assert hits, f"input not found: {pattern} - attach the bundle dataset and the output of 'DentaMind external data training'"
    return Path(hits[0])

BUNDLE = one("kaggle/dentamind_kaggle.py").parent.parent
want = json.loads((BUNDLE / "BUNDLE_MANIFEST.json").read_text())["files"]
for f in ("kaggle/dentamind_kaggle.py", "scripts/dataset/dsutil.py", "scripts/dataset/download_external.py", "scripts/dataset/remote_zip_list.py",
          "data/splits/combined_validation.csv", "docs/audit_evidence/ml_improvement/crop_robustness/val_detector_boxes.json"):
    assert hashlib.sha256((BUNDLE / f).read_bytes()).hexdigest() == want[f], f"bundle file changed: {f}"
print("bundle:", BUNDLE, "- files used verified against BUNDLE_MANIFEST.json")

RUNS = one("dentamind_artifacts/runs/BASE_s42/best.pth").parent.parent
ART = RUNS.parent
print("previous run artifacts:", ART)

CODE = Path("/kaggle/working/code")
CODE.mkdir(parents=True, exist_ok=True)
SRC = "#!/usr/bin/env python\n\"\"\"5-seed BASE ensemble (mean softmax probability) on the DENTEX validation split. Protocol: reports/ensemble_protocol.md.\n\n    .venv\\\\Scripts\\\\python.exe kaggle\\\\ensemble_eval.py\n\nReuses the Kaggle notebook's own crop builders, prediction and metric code (kaggle/dentamind_kaggle.py) so every number uses the same\ndefinitions as the BASE comparison. The DENTEX test split is never loaded. Writes reports/ensemble_evaluation.json and .md.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport sys\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\n\nHERE = Path(__file__).resolve().parent\nROOT = HERE.parent\nsys.path.insert(0, str(HERE))\nimport dentamind_kaggle as DK  # noqa: E402\nfrom dsutil import CLASSES  # noqa: E402\n\nRUN_DIR = HERE / \"artifacts_external_run1\"\nCK_DIR = RUN_DIR / \"base_checkpoints\"\n# protocol section 1: fixed members, identified by full SHA-256 (checkpoint_manifest.json of the Kaggle run)\nMEMBERS = {\n    42: \"0846c7595f9e567960d223988a284e8aab552b7a53be172f719e02508117c6ad\",\n    43: \"48c2b6718afdef0e273598e60b8feb51f3129db3d11da4d9f9c790cc2eb779cf\",\n    44: \"270cb3f190b99a9d56bc3c086a56223264e69bdc9ad989e77fe765d3ceb75a4b\",\n    45: \"05ed85068fec18102e5686d619488c50aef0fda69f4046a842dfe26cd62e22f8\",\n    46: \"653eaab4cfea1efa0b1bc3de683c707f28128aed97f9eee50619baa9ec59a30e\",\n}\nREPRO_TOL = 0.005            # protocol section 4\nBOOT_N, BOOT_SEED = 2000, 0  # protocol section 5 (reported, not used for acceptance)\n\n\nclass EnsembleError(RuntimeError):\n    pass\n\n\n# ------------------------------------------------------------------------------------------------------------- members\ndef sha256(path) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as fh:\n        for b in iter(lambda: fh.read(1 << 20), b\"\"):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef verify_members(ck_paths, members=MEMBERS) -> dict:\n    \"\"\"ck_paths: a folder holding BASE_s<seed>.pth, or {seed: path}. Returns {seed: path}. Stops on a missing file or a hash that\n    differs from the registered one (never substitutes another file).\"\"\"\n    if not isinstance(ck_paths, dict):\n        ck_paths = {s: Path(ck_paths) / f\"BASE_s{s}.pth\" for s in members}\n    out, problems = {}, []\n    for seed, want in sorted(members.items()):\n        p = Path(ck_paths[seed]) if ck_paths.get(seed) else Path(f\"<no path for seed {seed}>\")\n        if not p.is_file():\n            problems.append(f\"BASE_s{seed}: missing ({p})\")\n            continue\n        got = sha256(p)\n        if got != want:\n            problems.append(f\"BASE_s{seed}: sha256 {got[:16]}... != registered {want[:16]}...\")\n            continue\n        out[seed] = p\n    if problems:\n        raise EnsembleError(\"ensemble members not usable: \" + \"; \".join(problems))\n    return out\n\n\ndef check_class_order(class_names) -> None:\n    if list(class_names) != list(CLASSES):\n        raise EnsembleError(f\"class order {list(class_names)} differs from {list(CLASSES)}\")\n\n\ndef load_member(path, torch, tv, device):\n    \"\"\"Same construction as evaluate_checkpoint; strict load; output layer must have one unit per class.\"\"\"\n    m = DK.new_model(torch, tv, pretrained=False)\n    sd = torch.load(path, map_location=\"cpu\", weights_only=True)\n    if tuple(sd[\"fc.weight\"].shape) != (len(CLASSES), m.fc.in_features):\n        raise EnsembleError(f\"{Path(path).name}: output layer {tuple(sd['fc.weight'].shape)} does not match {len(CLASSES)} classes\")\n    m.load_state_dict(sd, strict=True)\n    return m.to(device).eval()\n\n\n# ------------------------------------------------------------------------------------------------------------- ensemble\ndef average_probs(prob_list) -> np.ndarray:\n    \"\"\"Arithmetic mean of member softmax outputs (protocol section 2). All members must give the same (N, C) shape, C = len(CLASSES),\n    finite values, rows summing to 1.\"\"\"\n    if not prob_list:\n        raise EnsembleError(\"no member probabilities\")\n    arrs = [np.asarray(p, dtype=np.float64) for p in prob_list]\n    shape = arrs[0].shape\n    if len(shape) != 2 or shape[1] != len(CLASSES):\n        raise EnsembleError(f\"member output shape {shape} is not (N, {len(CLASSES)})\")\n    for i, a in enumerate(arrs):\n        if a.shape != shape:\n            raise EnsembleError(f\"member {i} output shape {a.shape} differs from {shape}\")\n        if not np.isfinite(a).all() or (a < 0).any() or not np.allclose(a.sum(1), 1.0, atol=1e-4):\n            raise EnsembleError(f\"member {i} output is not a probability distribution per row\")\n    return np.mean(np.stack(arrs), axis=0)\n\n\ndef macro_f1(y, pred, n=len(CLASSES)) -> float:\n    \"\"\"Same definition as DK.report (classes with no prediction count as F1 0), vectorised for the bootstrap.\"\"\"\n    cm = np.bincount(np.asarray(y) * n + np.asarray(pred), minlength=n * n).reshape(n, n)\n    tp, pp, sup = np.diag(cm), cm.sum(0), cm.sum(1)\n    prec = np.divide(tp, pp, out=np.zeros(n), where=pp > 0)\n    rec = np.divide(tp, sup, out=np.zeros(n), where=sup > 0)\n    return float(np.divide(2 * prec * rec, prec + rec, out=np.zeros(n), where=(prec + rec) > 0).mean())\n\n\ndef bootstrap_diff(y, ens_pred, member_preds, n_boot=BOOT_N, seed=BOOT_SEED) -> dict:\n    \"\"\"95 % percentile interval of macro-F1(ensemble) - mean_k macro-F1(member k), resampling findings with replacement.\"\"\"\n    rng = np.random.default_rng(seed)\n    y, ens_pred = np.asarray(y), np.asarray(ens_pred)\n    diffs = np.empty(n_boot)\n    for b in range(n_boot):\n        i = rng.integers(0, len(y), len(y))\n        diffs[b] = macro_f1(y[i], ens_pred[i]) - np.mean([macro_f1(y[i], np.asarray(p)[i]) for p in member_preds])\n    return {\"n_boot\": n_boot, \"seed\": seed, \"mean\": float(diffs.mean()), \"ci95\": [float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))],\n            \"share_positive\": float((diffs > 0).mean())}\n\n\ndef ensemble_acceptance(ens: dict, members: dict, setting=\"det_matched\") -> dict:\n    \"\"\"Protocol section 5. ens = {\"gt\": report, \"det_matched\": report}; members = {seed: same}. Thresholds from DK.ACCEPT.\"\"\"\n    A = DK.ACCEPT\n    seeds = sorted(members)\n    if len(seeds) < A[\"seed_pairs_min\"]:\n        return {\"pass\": False, \"reason\": f\"only {len(seeds)} members\"}\n    mean = lambda f: float(np.mean([f(members[s]) for s in seeds]))  # noqa: E731\n    f1 = lambda r: r[setting][\"macro_f1\"]  # noqa: E731\n    pc = lambda c, m: mean(lambda r: r[setting][\"per_class\"][c][m])  # noqa: E731\n    gain = ens[setting][\"macro_f1\"] - mean(f1)\n    beats = int(sum(ens[setting][\"macro_f1\"] > f1(members[s]) for s in seeds))\n    e = lambda c, m: ens[setting][\"per_class\"][c][m]  # noqa: E731\n    c1 = gain > A[\"gain_min\"] and beats >= A[\"seed_pairs_min\"]\n    c2 = all(e(c, \"recall\") >= pc(c, \"recall\") for c in DK.MINORITY)\n    c3 = all(e(c, \"f1\") >= pc(c, \"f1\") - A[\"class_f1_drop_max\"] for c in CLASSES)\n    gt_delta = ens[\"gt\"][\"macro_f1\"] - mean(lambda r: r[\"gt\"][\"macro_f1\"])\n    c4 = gt_delta >= -A[\"gt_macro_f1_drop_max\"]\n    c5 = all(e(c, \"precision\") >= pc(c, \"precision\") - A[\"minority_precision_drop_max\"] for c in DK.MINORITY)\n    return {\"setting\": setting, \"members\": seeds, \"macro_f1_gain_vs_base_mean\": gain, \"members_beaten\": beats,\n            \"minority_recall_delta\": {c: e(c, \"recall\") - pc(c, \"recall\") for c in DK.MINORITY},\n            \"minority_precision_delta\": {c: e(c, \"precision\") - pc(c, \"precision\") for c in DK.MINORITY},\n            \"class_f1_delta\": {c: e(c, \"f1\") - pc(c, \"f1\") for c in CLASSES}, \"gt_macro_f1_delta\": gt_delta,\n            \"criteria\": {\"1_gain_gt_0.027_and_beats_4_of_5\": bool(c1), \"2_minority_recall_not_lower\": bool(c2),\n                         \"3_no_class_f1_drop_gt_0.05\": bool(c3), \"4_gt_macro_f1_drop_le_0.02\": bool(c4),\n                         \"5_minority_precision_drop_le_0.10\": bool(c5)},\n            \"pass\": bool(c1 and c2 and c3 and c4 and c5)}\n\n\ndef reproduction_check(local: dict, kaggle: dict, tol=REPRO_TOL) -> dict:\n    \"\"\"Protocol section 4: every member's local macro-F1 (gt and det_matched) must match the Kaggle run within tol.\"\"\"\n    rows, ok = {}, True\n    for s in sorted(local):\n        r = {}\n        for setting in (\"gt\", \"det_matched\"):\n            a, b = local[s][setting][\"macro_f1\"], kaggle[str(s)][setting][\"macro_f1\"]\n            r[setting] = {\"local\": a, \"kaggle\": b, \"abs_diff\": abs(a - b)}\n            ok &= abs(a - b) <= tol\n        rows[s] = r\n    return {\"tolerance\": tol, \"pass\": bool(ok), \"members\": rows}\n\n\n# ------------------------------------------------------------------------------------------------------------- latency\ndef latency(torch, models, X, device, reps=5):\n    \"\"\"Seconds per crop for one member and for the full ensemble; batch 64 and batch 1; after warm-up.\"\"\"\n    def timed(fn):\n        fn()                                   # warm-up\n        if device.type == \"cuda\":\n            torch.cuda.synchronize()\n        t = time.perf_counter()\n        for _ in range(reps):\n            fn()\n        if device.type == \"cuda\":\n            torch.cuda.synchronize()\n        return (time.perf_counter() - t) / reps\n    xb, x1 = X[:64], X[:1]\n    one = models[:1]\n    run = lambda ms, x: [DK.predict(torch, m, x, device) for m in ms]  # noqa: E731\n    out = {\"device\": torch.cuda.get_device_name(0) if device.type == \"cuda\" else \"cpu\", \"reps\": reps,\n           \"single_member_batch64_s_per_crop\": timed(lambda: run(one, xb)) / len(xb),\n           \"ensemble_batch64_s_per_crop\": timed(lambda: run(models, xb)) / len(xb),\n           \"single_member_batch1_s\": timed(lambda: run(one, x1)),\n           \"ensemble_batch1_s\": timed(lambda: run(models, x1))}\n    out[\"ratio_batch64\"] = out[\"ensemble_batch64_s_per_crop\"] / out[\"single_member_batch64_s_per_crop\"]\n    out[\"ratio_batch1\"] = out[\"ensemble_batch1_s\"] / out[\"single_member_batch1_s\"]\n    return out\n\n\n# ------------------------------------------------------------------------------------------------------------- main\ndef local_config() -> dict:\n    \"\"\"Paths in this repository. The Kaggle notebook (kaggle/make_ensemble_notebook.py) passes its own input locations instead.\"\"\"\n    return {\"ck_paths\": CK_DIR, \"class_meta\": RUN_DIR / \"checkpoints/baseline__BASE_s43.json\",\n            \"run_records\": {s: RUN_DIR / f\"runs/BASE_s{s}.json\" for s in MEMBERS}, \"kaggle_eval\": RUN_DIR / \"validation_eval.json\",\n            \"data_root\": ROOT / \"Data\", \"split_dir\": ROOT / \"data/splits\",\n            \"det_cache\": ROOT / \"docs/audit_evidence/ml_improvement/crop_robustness/val_detector_boxes.json\",\n            \"out_dir\": ROOT / \"reports\", \"environment\": \"local\"}\n\n\ndef main(cfg: dict | None = None):\n    import torch\n    import torchvision as tv\n    cfg = cfg or local_config()\n    t0 = time.time()\n    paths = verify_members(cfg[\"ck_paths\"])\n    check_class_order(json.loads(Path(cfg[\"class_meta\"]).read_text(encoding=\"utf-8\"))[\"class_names\"])\n    for s in MEMBERS:                         # every member comes from the same BASE configuration of the same run\n        r = json.loads(Path(cfg[\"run_records\"][s]).read_text(encoding=\"utf-8\"))\n        if r[\"experiment\"] != \"BASE\" or r[\"checkpoint_sha256\"] != MEMBERS[s]:\n            raise EnsembleError(f\"run record BASE_s{s} does not describe the registered checkpoint\")\n\n    data_root = Path(cfg[\"data_root\"])\n    rows = DK.load_split(cfg[\"split_dir\"], \"validation\")       # validation only; the test split is never read\n    n_ok, problems = DK.verify_images(rows, data_root)\n    if problems:\n        raise EnsembleError(f\"validation images not byte-identical to the manifest: {problems}\")\n    Xg, y, _ = DK.build_crops(rows, data_root)\n    Xd, mi, missed = DK.build_detector_crops(rows, data_root, cfg[\"det_cache\"], lambda r: r[\"project_image_id\"])\n    print(f\"validation: {len(y)} findings, {n_ok} radiographs, detector-matched {len(mi)}, unmatched {missed}\", flush=True)\n\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    probs = {}\n    for s, p in paths.items():\n        m = load_member(p, torch, tv, device)\n        probs[s] = {\"gt\": DK.predict(torch, m, Xg, device), \"det\": DK.predict(torch, m, Xd, device)}\n        del m\n        if device.type == \"cuda\":\n            torch.cuda.empty_cache()\n        print(f\"member s{s} predicted\", flush=True)\n    yd = y[mi]\n    members = {s: {\"gt\": DK.report(y, pr[\"gt\"].argmax(1), pr[\"gt\"]), \"det_matched\": DK.report(yd, pr[\"det\"].argmax(1), pr[\"det\"])}\n               for s, pr in probs.items()}\n\n    kaggle_eval = json.loads(Path(cfg[\"kaggle_eval\"]).read_text(encoding=\"utf-8\"))[\"eval\"][\"BASE\"]\n    repro = reproduction_check(members, kaggle_eval)\n    result = {\"protocol\": \"reports/ensemble_protocol.md\", \"generated_utc\": datetime.now(timezone.utc).isoformat(),\n              \"environment\": cfg.get(\"environment\"), \"script_sha256\": sha256(__file__),\n              \"members\": {s: {\"sha256\": MEMBERS[s], \"path\": str(paths[s])} for s in MEMBERS},\n              \"data\": {\"split\": \"data/splits/combined_validation.csv\", \"findings\": int(len(y)), \"radiographs\": int(n_ok),\n                       \"detector_matched\": int(len(mi)), \"detector_unmatched\": int(missed), \"test_split_loaded\": False},\n              \"software\": DK.software(), \"reproduction_check\": repro, \"member_results\": members}\n    out_dir = Path(cfg[\"out_dir\"])\n    out_dir.mkdir(parents=True, exist_ok=True)\n    out_json = out_dir / (\"ensemble_evaluation.json\" if cfg.get(\"environment\") == \"local\" else \"ensemble_evaluation_kaggle.json\")\n    if not repro[\"pass\"]:\n        result[\"stopped\"] = \"reproduction check failed (protocol section 4); no ensemble comparison made\"\n        out_json.write_text(json.dumps(result, indent=1, default=float), encoding=\"utf-8\")\n        raise SystemExit(result[\"stopped\"])\n\n    pg = average_probs([probs[s][\"gt\"] for s in sorted(probs)])\n    pd = average_probs([probs[s][\"det\"] for s in sorted(probs)])\n    ens = {\"gt\": DK.report(y, pg.argmax(1), pg), \"det_matched\": DK.report(yd, pd.argmax(1), pd)}\n    result[\"ensemble\"] = ens\n    result[\"acceptance\"] = ensemble_acceptance(ens, members)\n    result[\"bootstrap_det_matched\"] = bootstrap_diff(yd, pd.argmax(1), [probs[s][\"det\"].argmax(1) for s in sorted(probs)])\n    result[\"bootstrap_gt\"] = bootstrap_diff(y, pg.argmax(1), [probs[s][\"gt\"].argmax(1) for s in sorted(probs)])\n\n    models = [load_member(paths[s], torch, tv, device) for s in sorted(paths)]\n    result[\"latency\"] = latency(torch, models, Xd, device)\n    del models\n    result[\"runtime_s\"] = round(time.time() - t0, 1)\n    out_json.write_text(json.dumps(result, indent=1, default=float), encoding=\"utf-8\")\n    print(json.dumps({\"reproduction_pass\": repro[\"pass\"], \"acceptance\": result[\"acceptance\"][\"criteria\"],\n                      \"pass\": result[\"acceptance\"][\"pass\"]}, indent=1))\n    print(\"wrote\", out_json)\n\n\nif __name__ == \"__main__\":\n    main()\n"
assert hashlib.sha256(SRC.encode("utf-8")).hexdigest() == "44a89511641a7c23ffe160eb30c20958191738407ff95f0ea283d932c3cee262", "embedded script differs from kaggle/ensemble_eval.py"
(CODE / "ensemble_eval.py").write_text(SRC, encoding="utf-8")
for p in (CODE, BUNDLE / "kaggle", BUNDLE / "scripts/dataset"):
    sys.path.insert(0, str(p))
import dentamind_kaggle as DK
import download_external as DX
import ensemble_eval as EV
print("ensemble_eval.py sha256 44a89511641a7c23ffe160eb30c20958191738407ff95f0ea283d932c3cee262")

In [ ]:
# ---- 2. DENTEX validation images (official archive, verified per image by SHA-256) -----------------------------------
DATA_ROOT = Path("/tmp/dentamind_data")
rows = DK.load_split(BUNDLE / "data/splits", "validation")
if DK.verify_images(rows, DATA_ROOT)[1]:
    t = time.time()
    DX.fetch_dentex_disease_subset(DATA_ROOT)
    print(f"fetched DENTEX disease subset in {time.time() - t:.0f}s")
n_ok, problems = DK.verify_images(rows, DATA_ROOT)
print(f"validation radiographs verified: {n_ok}, problems: {problems}")
assert not problems

In [ ]:
# ---- 3. run the registered evaluation (reproduction gate first, then ensemble vs members) ------------------------------
cfg = {"ck_paths": {s: RUNS / f"BASE_s{s}" / "best.pth" for s in EV.MEMBERS},
       "class_meta": RUNS / "BASE_s43" / "best.json",
       "run_records": {s: RUNS / f"BASE_s{s}" / "result.json" for s in EV.MEMBERS},
       "kaggle_eval": ART / "validation_eval.json",
       "data_root": DATA_ROOT, "split_dir": BUNDLE / "data/splits",
       "det_cache": BUNDLE / "docs/audit_evidence/ml_improvement/crop_robustness/val_detector_boxes.json",
       "out_dir": Path("/kaggle/working/ensemble"), "environment": "kaggle"}
try:
    EV.main(cfg)
except SystemExit as e:
    print("STOPPED:", e)
out = Path("/kaggle/working/ensemble/ensemble_evaluation_kaggle.json")
print(out, out.stat().st_size if out.is_file() else "missing")